In [4]:
pip install pandas scikit-learn seaborn matplotlib numpy

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 9.9 MB/s  0:00:00m eta 0:00:01
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 6.8 MB/s  0:00:01 eta 0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 6.5 MB/s  0:00:00 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 7.2 MB/s  0:00:00 eta 0:00:01
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 6.4 MB/s  0:00:03 eta 0:00:01
Using cached threadpoolctl-3.6.

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error

**Predicting a song's streams based on musical features**

In [21]:
df = pd.read_csv("spotify_alltime_top100_songs.csv")

In [22]:
df.head()

,alltime_rank,song_title,artist,total_streams_billions,primary_genre,bpm,release_year,artist_country,explicit,danceability,energy,valence,acousticness,dataset_part
0,1,Blinding Lights,The Weeknd,5.26,Synth-Pop,171,2019,Canada,False,0.51,0.80,0.33,0.00,Spotify All-Time Most Streamed Top 100
1,2,Shape of You,Ed Sheeran,4.90,Pop/Dancehall,96,2017,UK,False,0.83,0.65,0.93,0.08,Spotify All-Time Most Streamed Top 100
2,3,Someone You Loved,Lewis Capaldi,4.05,Pop,77,2018,UK,False,0.60,0.45,0.42,0.29,Spotify All-Time Most Streamed Top 100
3,4,Sunflower,Post Malone & Swae Lee,3.98,Hip-Hop/Pop,93,2018,USA,False,0.76,0.49,0.84,0.15,Spotify All-Time Most Streamed Top 100
4,5,One Dance,Drake,3.92,Afrobeats/Pop,100,2016,Canada,False,0.79,0.62,0.68,0.09,Spotify All-Time Most Streamed Top 100


In [23]:
df.shape

(100, 14)

In [31]:
df.isna().sum()

alltime_rank              0
song_title                0
artist                    0
total_streams_billions    0
primary_genre             0
bpm                       0
release_year              0
artist_country            0
explicit                  0
danceability              0
energy                    0
valence                   0
acousticness              0
dtype: int64

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   alltime_rank            100 non-null    int64  
 1   song_title              100 non-null    str    
 2   artist                  100 non-null    str    
 3   total_streams_billions  100 non-null    float64
 4   primary_genre           100 non-null    str    
 5   bpm                     100 non-null    int64  
 6   release_year            100 non-null    int64  
 7   artist_country          100 non-null    str    
 8   explicit                100 non-null    int64  
 9   danceability            100 non-null    float64
 10  energy                  100 non-null    float64
 11  valence                 100 non-null    float64
 12  acousticness            100 non-null    float64
dtypes: float64(5), int64(4), str(4)
memory usage: 13.9 KB


In [24]:
df.drop('dataset_part',axis=1, inplace=True)

In [38]:
# df['explicit'] = df['explicit'].map({False:0,True:1})

In [39]:
df['explicit'] = df['explicit'].astype(int)

In [40]:
df['explicit']

0     0
1     0
2     0
3     0
4     0
     ..
95    0
96    0
97    0
98    1
99    0
Name: explicit, Length: 100, dtype: int64

In [64]:
num_cols = [col for col in df.columns if df[col].dtype != 'str']

In [67]:
#num_cols = df.select_dtypes(include='number').columns

In [68]:
num_cols

['alltime_rank',
 'total_streams_billions',
 'bpm',
 'release_year',
 'explicit',
 'danceability',
 'energy',
 'valence',
 'acousticness']

In [69]:
X = df[num_cols]

In [71]:
y = df['total_streams_billions']

In [72]:
y

0     5.26
1     4.90
2     4.05
3     3.98
4     3.92
      ... 
95    1.92
96    1.44
97    1.56
98    3.05
99    1.60
Name: total_streams_billions, Length: 100, dtype: float64

In [73]:
X_train , X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3 , random_state = 42)

In [80]:
X.shape ,X_train.shape, X_test.shape

((100, 9), (70, 9), (30, 9))

In [84]:
y.shape, y_train.shape, y_test.shape

((100,), (70,), (30,))